In [1]:
import os
import json
import pandas as pd
# from datetime import datetime
from dotenv import load_dotenv

from options import OptionSurface, Deribit, OKX, Bybit
from portfolio_management import Portfolio
from api_client import TradingDeskAPI
from scanner import MarketScanner

In [2]:
# Initialize all classes and parameters

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
# load_dotenv(r"C:/Users/brian/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
FRACTION = float(os.getenv("FRACTION"))
MAX_POSITION = float(os.getenv("MAX_POSITION"))
ENTRY_EV_THRESHOLD = float(os.getenv("ENTRY_EV_THRESHOLD")) # require 1% edge, default = 0
EXIT_EV_THRESHOLD = float(os.getenv("EXIT_EV_THRESHOLD"))
print(BASE_URL)

with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
    crypto_tag_ids = set(json.load(f))

s = OptionSurface()
portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScanner(api=api)

# target_expiry_str = "25DEC26"
# target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]
# currencies = ["BTC", "ETH", "HYPE", "SOL", "ZEC"]

https://alphasignal-dev.moretoncp.com


In [3]:
# 1. Get all dfs
orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet") # will be overwritten
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

# fills_df = fills_df.iloc[0:0]

In [4]:
# # # BACKTEST

# # 1. Get all dfs
# orders_df_backtest = pd.read_parquet(f"{DATA_DIR}/orders_backtest.parquet")
# fills_df_backtest = pd.read_parquet(f"{DATA_DIR}/fills_backtest.parquet")
# positions_df_backtest = pd.read_parquet(f"{DATA_DIR}/positions_backtest.parquet")
# realized_pnl_df_backtest = pd.read_parquet(f"{DATA_DIR}/realized_pnl_backtest.parquet") # will be overwritten
# equity_df_backtest = pd.read_parquet(f"{DATA_DIR}/equity_backtest.parquet")
# cash_df_backtest = pd.read_parquet(f"{DATA_DIR}/cash_backtest.parquet")

# cash = cash_df_backtest.iloc[-1]["balance"]

# fills_df_backtest = portfolio.sync_fills_test(api=api, orders_df_backtest=orders_df_backtest, fills_df_backtest=fills_df_backtest)
# positions_df_backtest, realized_pnl_df_backtest = portfolio.reconstruct_positions_fifo(fills_df=fills_df_backtest)
# positions_df_backtest = portfolio.mark_positions_to_market(api=api, positions_df=positions_df_backtest)
# equity_df_backtest = portfolio.calculate_equity_test(api=api, positions_df=positions_df_backtest, 
#                         realized_pnl_df=realized_pnl_df_backtest, equity_df=equity_df_backtest, cash=cash)

In [4]:
# 2. Get newly executed trades
fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)
fills_df

,question,order_id,condition_id,token_id,outcome,side,price,shares,fee,timestamp,fill_id
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,0.00635,2026-08-14 08:25:11+00:00,4c3c68e2-7b89-463c-ae03-48b92808eeef
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,0.01330,2026-08-19 08:00:45+00:00,ff80c017-a13d-43ee-85c0-35a2c289c1b6
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,0.01032,2026-08-19 08:01:06+00:00,cd0ebe33-7e52-44c6-a022-466914d90934


In [5]:
# 3. Reconstruct portfolio
# 4. Calculate realized P&L
positions_df, realized_pnl_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df) # new realized_df to overwrite old
realized_pnl_df

,question,condition_id,token_id,outcome,realized_shares,realized_pnl,realized_fees
0,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,0.0,0.0,0.0
1,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,0.0,0.0,0.0
2,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,0.0,0.0,0.0


In [6]:
# 5. Sync positions with api
portfolio.reconcile_positions(api=api, positions_df=positions_df)

POSITION SYNCED: 4251240413067872674686064123803499349976238079777545616008767482027755117695
POSITION SYNCED: 447913121087537662187327157218300928168985707076581166939650812231975265268
POSITION SYNCED: 96993471854400156408670527613150944443359272190785251193551242374636006072800


In [7]:
# 6. Get latest order state
orders_df = portfolio.sync_orders(api=api, orders_df=orders_df)
orders_df

,question,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.877934,GTC,OPEN,2026-08-14 09:28:01.978131+00:00,NaT
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.959176,GTC,OPEN,2026-08-19 08:00:40.748876+00:00,NaT
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.913116,GTC,OPEN,2026-08-19 08:01:02.498064+00:00,NaT


In [8]:
# 7. Mark positions to market
positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)
positions_df

,question,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees,current_price,market_value,unrealized_pnl,unrealized_return
0,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.76530,0.962687,0.0,0.0,0.0,0.880,4.35600,-0.40930,-0.085892
1,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.76811,0.971102,0.0,0.0,0.0,0.940,4.61540,-0.15271,-0.032027
2,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.78382,0.982304,0.0,0.0,0.0,0.983,4.78721,0.00339,0.000709


In [9]:
# 8. Calculate equity
equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, 
                                realized_pnl_df=realized_pnl_df, equity_df=equity_df)
equity_df

Equity:  99.45
Return:  0.0604%
Sharpe: N/A
Sortino: N/A


,timestamp,cash,market_value,equity,realized_pnl,unrealized_pnl,daily_return,period_return,sharpe,sortino
0,2026-08-14 07:47:24.047885+00:00,100.00000,0.00000,100.00000,0.0,0.00000,NaN,NaN,NaN,NaN
1,2026-08-15 14:14:05.272655+00:00,95.21618,4.79695,100.01313,0.0,0.01948,NaN,0.000131,NaN,NaN
2,2026-08-17 02:58:57.343900+00:00,95.21618,4.77747,99.99365,0.0,0.00000,NaN,-0.000195,NaN,NaN
3,2026-08-19 07:56:33.538278+00:00,95.21618,4.78234,99.99852,0.0,0.00487,NaN,0.000049,NaN,NaN
4,2026-08-19 08:05:37.263592+00:00,85.68277,14.23772,99.92049,0.0,-0.04954,NaN,-0.000780,NaN,NaN
5,2026-08-24 13:32:14.011333+00:00,85.68747,13.80372,99.49119,0.0,-0.51351,NaN,-0.004296,NaN,NaN
6,2026-08-25 03:29:31.510439+00:00,85.68857,13.71458,99.40315,0.0,-0.60265,NaN,-0.000885,NaN,NaN
7,2026-08-27 02:38:40.684140+00:00,85.69107,13.69973,99.39080,0.0,-0.61750,NaN,-0.000124,NaN,NaN
8,2026-08-28 08:23:58.626786+00:00,85.69227,13.75861,99.45088,0.0,-0.55862,NaN,0.000604,NaN,NaN


In [10]:
# 9. Get current markets
all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)
all_markets_df.head()

Fetched 100 markets | Total: 100
Fetched 100 markets | Total: 200
Fetched 100 markets | Total: 300
Fetched 100 markets | Total: 400
Fetched 100 markets | Total: 500
Fetched 100 markets | Total: 600
Fetched 100 markets | Total: 700
Fetched 100 markets | Total: 800
Fetched 100 markets | Total: 900
Fetched 100 markets | Total: 1,000


,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,version,negRiskMarketID,seriesColor,showGmpSeries,showGmpOutcome,oneHourPriceChange,umaResolutionStatus,eventStartTime,gameStartTime,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2027-01-01T04:59:00Z,178099.62115,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,336757.39998,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,351668.08971,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,518563.90542,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,473233.39836,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN


In [11]:
all_markets_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,version,negRiskMarketID,seriesColor,showGmpSeries,showGmpOutcome,oneHourPriceChange,umaResolutionStatus,eventStartTime,gameStartTime,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2027-01-01T04:59:00Z,178099.62115,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,336757.39998,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,351668.08971,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,518563.90542,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,473233.39836,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1125396,"Will S&P 500 (SPX) hit $4,500 (LOW) in December?",0xdda8b52c6bf2cf875f57572345caca0c145924d782b9...,spx-hit-4500-low-dec-2026,https://finance.yahoo.com/quote/%5EGSPC/,2026-12-31T21:00:00Z,22737.15084,2026-01-07T02:33:16.546561Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
996,1125397,"Will S&P 500 (SPX) close at <$6,000 in December?",0xd9bdd7fbf3ba219c649424fa62180bfe139865ade439...,spx-close-below-6000-dec-2026,https://finance.yahoo.com/quote/%5EGSPC/history,2026-12-31T21:00:00Z,17242.2457,2026-01-07T02:24:11.422854Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0xbf17bd440e263ff25b13e6dbd0a49699fadd457e3113...,,False,False,NaN,NaN,NaN,NaN,NaN
997,1126504,Will the US acquire part of Greenland in 2026?,0x890fc3ba40458db0b67560691aa1597344c2a0560d18...,will-the-us-acquire-any-part-of-greenland-in-2026,,2027-01-01T04:59:00Z,45622.1248,2026-01-07T04:34:36.508662Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
998,1129894,Will United Russia (ER) win the most seats in ...,0x94ffc785f73c9d70c0c9550b2412842d025d20d02f9e...,will-united-russia-er-win-the-most-seats-in-th...,NaN,2026-09-30T00:00:00Z,107575.45443,2026-01-07T21:27:45.212078Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x5dd3c57e95e4b3d29722ae868668bc2e0848028a1d26...,NaN,False,False,NaN,NaN,NaN,NaN,NaN


In [13]:
# 10. Initialize variance surface
deribit = Deribit(currencies=currencies)
okx = OKX(currencies=currencies)
bybit = Bybit(currencies=currencies)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

currency: BTC, spot: 79763.0, volume24h: 17.339
currency: ETH, spot: 2497.1, volume24h: 72.0259
currency: BTC, spot: 79819.9, volume24h: 7279.35222265
currency: ETH, spot: 2499.84, volume24h: 129692.832789
currency: BTC, spot: 79818.3, volume24h: 9091.065024
currency: ETH, spot: 2499.73, volume24h: 96948.50016


In [14]:
# 11. Scan markets
markets_df, opportunities_df, arb_candidates_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)
opportunities_df.head()

question: Will Bitcoin hit $150k by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 150000.0
iv: 0.5168002998846805
buy_yes_ev: -0.010858749432664797
sell_yes_ev: 0.0051954194326647996
buy_no_ev: 0.00519541943266482
sell_no_ev: -0.010858749432664783
buy_yes_kelly: -0.005640113320466644
sell_yes_kelly: 0.08194430055444447
buy_no_kelly: 0.08194430055444474
sell_no_kelly: -0.005640113320466637

question: Will Bitcoin reach $200,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 200000.0
iv: 0.5923153381240728
buy_yes_ev: -0.013215219583421568
sell_yes_ev: 0.009943369583421569
buy_no_ev: 0.009943369583421575
sell_no_ev: -0.01321521958342154
buy_yes_kelly: -0.006729890351523179
sell_yes_kelly: 0.33371670620534843
buy_no_kelly: 0.33371670620534843
sell_no_kelly: -0.006729890351523165

question: Will Bitcoin reach $190,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 190000.0
iv: 0.

,question,endDate,yes_ask,yes_bid,no_ask,no_bid,model_prob,id,conditionId,slug,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
600,"Will Ethereum dip to $1,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.153,0.139,0.861,0.847,0.216770,701552,0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,...,0.054698,-0.086147,-0.086147,0.054698,0.032639,-0.329756,-0.329756,0.032639,0.054698,buy_yes_ev
597,"Will Ethereum reach $4,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.12,0.11,0.89,0.88,0.075128,701547,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,will-ethereum-reach-4500-by-december-31-2026,...,-0.052264,0.028019,0.028019,-0.052264,-0.029947,0.135819,0.135819,-0.029947,0.028019,sell_yes_ev
595,"Will Ethereum reach $5,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.06,0.059,0.941,0.94,0.027720,701545,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,will-ethereum-reach-5500-by-december-31-2026,...,-0.036228,0.027394,0.027394,-0.036228,-0.019352,0.248522,0.248522,-0.019352,0.027394,buy_no_ev
599,"Will Ethereum reach $3,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.31,0.3,0.7,0.69,0.262151,701549,0x42945ea5657d6f6e77969a06661b29d6f6295083d1ef...,will-ethereum-reach-3500-by-december-31-2026,...,-0.062822,0.023149,0.023149,-0.062822,-0.046533,0.040569,0.040569,-0.046533,0.023149,sell_yes_ev
594,"Will Ethereum reach $6,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.058,0.042,0.958,0.942,0.018085,701544,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,will-ethereum-reach-6000-by-december-31-2026,...,-0.043739,0.021098,0.021098,-0.043739,-0.023311,0.269224,0.269224,-0.023311,0.021098,buy_no_ev


In [ ]:
#               Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.122	0.121	0.879	0.878	0.219853	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.090355	-0.106299	-0.106299	0.090355	0.051898	-0.468050	-0.468050	0.051898	0.090355	buy_yes_ev

# 06	701552	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.146	0.89	0.264426	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...		59292.16479	...	0.109698	-0.161279	-0.161279	0.109698	0.064889	-0.781790	-0.781790	0.064889	0.109698	buy_yes_ev

# 585	701552	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.402	0.599	0.525600	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...		80206.24887	...	0.106772	-0.141414	-0.141414	0.106772	0

In [15]:
opportunities_df

,question,endDate,yes_ask,yes_bid,no_ask,no_bid,model_prob,id,conditionId,slug,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
600,"Will Ethereum dip to $1,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.153,0.139,0.861,0.847,0.216770,701552,0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,...,0.054698,-0.086147,-0.086147,0.054698,0.032639,-0.329756,-0.329756,0.032639,0.054698,buy_yes_ev
597,"Will Ethereum reach $4,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.12,0.11,0.89,0.88,0.075128,701547,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,will-ethereum-reach-4500-by-december-31-2026,...,-0.052264,0.028019,0.028019,-0.052264,-0.029947,0.135819,0.135819,-0.029947,0.028019,sell_yes_ev
595,"Will Ethereum reach $5,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.06,0.059,0.941,0.94,0.027720,701545,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,will-ethereum-reach-5500-by-december-31-2026,...,-0.036228,0.027394,0.027394,-0.036228,-0.019352,0.248522,0.248522,-0.019352,0.027394,buy_no_ev
599,"Will Ethereum reach $3,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.31,0.3,0.7,0.69,0.262151,701549,0x42945ea5657d6f6e77969a06661b29d6f6295083d1ef...,will-ethereum-reach-3500-by-december-31-2026,...,-0.062822,0.023149,0.023149,-0.062822,-0.046533,0.040569,0.040569,-0.046533,0.023149,sell_yes_ev
594,"Will Ethereum reach $6,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.058,0.042,0.958,0.942,0.018085,701544,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,will-ethereum-reach-6000-by-december-31-2026,...,-0.043739,0.021098,0.021098,-0.043739,-0.023311,0.269224,0.269224,-0.023311,0.021098,buy_no_ev
593,"Will Ethereum reach $6,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.04,0.033,0.967,0.96,0.011674,701543,0x0f0499d1049385b1d53ffee6c42a1de7424e551c7652...,will-ethereum-reach-6500-by-december-31-2026,...,-0.031014,0.019093,0.019093,-0.031014,-0.016199,0.310285,0.310285,-0.016199,0.019093,buy_no_ev
592,"Will Ethereum reach $7,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.03,0.029,0.971,0.97,0.008139,701542,0x4ee2c6081e18cc0d1f3347f6219940bda986657f1d15...,will-ethereum-reach-7000-by-december-31-2026,...,-0.023898,0.018890,0.018890,-0.023898,-0.012345,0.349440,0.349440,-0.012345,0.018890,sell_yes_ev
591,"Will Ethereum reach $7,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.024,0.023,0.977,0.976,0.006467,701541,0x4fc8725f00ba06576d7c78131e4589cc93a4e133128e...,will-ethereum-reach-7500-by-december-31-2026,...,-0.019173,0.014960,0.014960,-0.019173,-0.009839,0.349103,0.349103,-0.009839,0.014960,buy_no_ev
868,"Will Bitcoin reach $250,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.017,0.016,0.984,0.983,0.001304,1057883,0x6fefc0438c7598b23531457c8c60541990d0786bd4bd...,will-bitcoin-reach-250000-by-december-31-2026-...,...,-0.016866,0.013594,0.013594,-0.016866,-0.008589,0.456229,0.456229,-0.008589,0.013594,buy_no_ev
590,"Will Ethereum reach $8,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.021,0.019,0.981,0.979,0.004686,701540,0xac0b4316113b50e0e667918aee72fa0eb006342245a5...,will-ethereum-reach-8000-by-december-31-2026,...,-0.017753,0.013009,0.013009,-0.017753,-0.009080,0.367588,0.367588,-0.009080,0.013009,buy_no_ev


In [16]:
cross_market_arb_df, vertical_arb_df = scanner.scan_arbitrage(arb_candidates_df)
vertical_arb_df

                                              question               endDate  \
165       Will Bitcoin hit $150k by December 31, 2026?  2027-01-01T05:00:00Z   
579  Will Bitcoin reach $150,000 by December 31, 2026?  2027-01-01T05:00:00Z   

    yes_ask yes_bid no_ask no_bid  model_prob      id  \
165   0.035   0.034  0.966  0.965    0.026506  573656   
579   0.035   0.031  0.969  0.965    0.026506  701491   

                                           conditionId  \
165  0x02deb9538f5c123373adaa4ee6217b01745f1662bc90...   
579  0xa7b594ae07d5c1590fa86028fcc2f870599043723741...   

                                                  slug  ... buy_yes_ev  \
165          will-bitcoin-hit-150k-by-december-31-2026  ...  -0.010859   
579  will-bitcoin-reach-150000-by-december-31-2026-...  ...  -0.010859   

    sell_yes_ev buy_no_ev sell_no_ev buy_yes_kelly sell_yes_kelly  \
165    0.005195  0.005195  -0.010859      -0.00564       0.081944   
579    0.002392  0.002392  -0.010859      -0.00564 

,currency,event_type,direction,lower_strike,lower_question,lower_id,lower_side,lower_price,lower_cost,higher_strike,higher_question,higher_id,higher_side,higher_price,higher_cost,total_cost,guaranteed_profit


In [17]:
# 12. manage cancel orders
orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
                ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)
orders_df

,question,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.877934,GTC,OPEN,2026-08-14 09:28:01.978131+00:00,NaT
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.959176,GTC,OPEN,2026-08-19 08:00:40.748876+00:00,NaT
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.913116,GTC,OPEN,2026-08-19 08:01:02.498064+00:00,NaT


In [18]:
# 13. Risk management
orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
            markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

pos: question              Will Ethereum reach $4,500 by December 31, 2026?
condition_id         0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...
token_id             4251240413067872674686064123803499349976238079...
outcome                                                             No
shares                                                            4.95
cost_basis                                                      4.7653
avg_entry_price                                               0.962687
realized_pnl                                                       0.0
realized_shares                                                    0.0
realized_fees                                                      0.0
current_price                                                     0.88
market_value                                                     4.356
unrealized_pnl                                                 -0.4093
unrealized_return                                            -0.085892
N

In [ ]:
# 14. New opportunities
orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
            ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

In [17]:
# # BACKTEST

# # 13. Risk management
# orders_df_backtest = portfolio.run_risk_management_test(api=api, positions_df_backtest=positions_df_backtest, 
#     markets_df=markets_df, orders_df_backtest=orders_df_backtest, cash=cash, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

# # 14. New opportunities
# orders_df_backtest = portfolio.run_new_opportunities_test(api=api, opportunities_df=opportunities_df, orders_df_backtest=orders_df_backtest,
#     cash=cash, ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

# cash_df_backtest = portfolio.update_cash(cash=cash, cash_df_backtest=cash_df_backtest)

# # # 15. Safe dfs
# portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
# portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
# portfolio.save(DATA_DIR, orders_df_backtest, "orders_backtest")
# portfolio.save(DATA_DIR, fills_df_backtest, "fills_backtest")
# portfolio.save(DATA_DIR, positions_df_backtest, "positions_backtest")
# portfolio.save(DATA_DIR, realized_pnl_df_backtest, "realized_pnl_backtest")
# portfolio.save(DATA_DIR, equity_df_backtest, "equity_backtest")

In [19]:
# # Initialize all classes and parameters

# # load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
# load_dotenv(r"C:/Users/brian/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
# BASE_URL = os.getenv("BASE_URL")
# USER_EMAIL = os.getenv("USER_EMAIL")
# USER_PASSWORD = os.getenv("USER_PASSWORD")
# JWT_TOKEN = os.getenv("JWT")
# DATA_DIR = os.getenv("DATA_DIR")
# FRACTION = float(os.getenv("FRACTION"))
# MAX_POSITION = float(os.getenv("MAX_POSITION"))
# ENTRY_EV_THRESHOLD = float(os.getenv("ENTRY_EV_THRESHOLD")) # require 1% edge, default = 0
# EXIT_EV_THRESHOLD = float(os.getenv("EXIT_EV_THRESHOLD"))
# print(BASE_URL)

# with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
#     crypto_tag_ids = set(json.load(f))

# s = OptionSurface()
# portfolio = Portfolio()
# api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
# scanner = MarketScanner(api=api)

# target_expiry_str = "25DEC26"
# target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

# currencies = ["BTC", "ETH"]
# # currencies = ["BTC", "ETH", "HYPE", "SOL", "ZEC"]

# for i in range(1):
#     # 1. Get all dfs
#     orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
#     fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
#     positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
#     realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet") # will be overwritten
#     equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

#     # 2. Get newly executed trades
#     fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)

#     # 3. Reconstruct portfolio
#     # 4. Calculate realized P&L
#     positions_df, realized_pnl_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df) # new realized_df to overwrite old

#     # 5. Sync positions with api
#     portfolio.reconcile_positions(api=api, positions_df=positions_df)

#     # 6. Get latest order state
#     orders_df = portfolio.sync_orders(api=api, orders_df=orders_df)

#     # 7. Mark positions to market
#     positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)

#     # 8. Calculate equity
#     equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, 
#                                     realized_pnl_df=realized_pnl_df, equity_df=equity_df)

#     # 9. Get current markets
#     all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)

#     # 10. Initialize variance surface
#     deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
#     okx = OKX(currencies=currencies, target_expiry=target_expiry)
#     bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

#     s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

#     # 11. Scan markets
#     markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)

#     # 12. manage cancel orders
#     orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
#                 ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

#     # 13. Risk management
#     orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
#             markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

#     # 14. New opportunities
#     orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
#             ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

#     # 15. Safe dfs
portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
portfolio.save(DATA_DIR, orders_df, "orders")
portfolio.save(DATA_DIR, fills_df, "fills")
portfolio.save(DATA_DIR, positions_df, "positions")
portfolio.save(DATA_DIR, realized_pnl_df, "realized_pnl")
portfolio.save(DATA_DIR, equity_df, "equity")

#     # time.sleep(300)

save_snapshots df saved at: data/20260828/markets_20260828_162510.parquet
save_snapshots df saved at: data/20260828/opportunities_20260828_162511.parquet
save df saved at:  data/orders.parquet
save df saved at:  data/fills.parquet
save df saved at:  data/positions.parquet
save df saved at:  data/realized_pnl.parquet
save df saved at:  data/equity.parquet


In [ ]:
# Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

# 1. Is the options-derived probability actually predictive?

# 2. Are you accounting for crypto risk premia / risk-neutral vs physical probabilities?

# 3. Are your touch probabilities correctly calibrated?

# 4. Are fees and prediction-market spreads killing the apparent EV?

# 5. Does your exit rule actually improve realized P&L?

# 6. Are multiple contracts giving you the same underlying exposure?

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4